# Lab 6: (Sub)Word embeddings
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer the questions in `Lab6.md`. Make sure to include in this notebook all the tests and experiments you run. Make sure to also cite any external resources you use. 

## Part 1: Computing similarity between `GLoVe` embeddings

### Part 1.1

In [1]:
import pickle
import numpy as np

with open('glove_dolma_300_10k.pkl', 'rb') as f:
    glove_embeddings = pickle.load(f)

In [2]:
print(len(glove_embeddings.keys()))

10000


In [3]:
words = ['hello', 'cat', 'chomsky', 'supercalifragilisticexpialidocious']

for word in words:
    if word in glove_embeddings:
        vec = glove_embeddings[word]
        print(f'"{word}": Dimensions', vec.shape, "Mean:", np.mean(vec))
    else:
        print(f'"{word}": Missing embedding')

"hello": Dimensions (300,) Mean: -5.767882e-05
"cat": Dimensions (300,) Mean: -0.00032740872
"chomsky": Missing embedding
"supercalifragilisticexpialidocious": Missing embedding


### Part 1.2

Implement and test the following functions

In [4]:
def get_word_vector_glove(word:str,embeddings:dict):
    """
    Return embedding of word if it exists, if not the mean embedding of all words
    """
    if word in embeddings:
        return embeddings[word]
    else:
        all_embeddings = np.array(list(embeddings.values()))
        return np.mean(all_embeddings, axis=0)

In [5]:
## Tests
print(str(get_word_vector_glove('hello',glove_embeddings).shape) == '(300,)')
print(str(np.mean(get_word_vector_glove('hello',glove_embeddings))) == '-5.767882e-05')
print(str(get_word_vector_glove('hello',glove_embeddings)[0]) == '0.200562')

print(str(get_word_vector_glove('supercalifragilisticexpialidocious',glove_embeddings).shape) == '(300,)')
print(str(np.mean(get_word_vector_glove('supercalifragilisticexpialidocious',glove_embeddings))) == '-0.0065580728')
print(str(get_word_vector_glove('supercalifragilisticexpialidocious',glove_embeddings)[0]) == '-0.08470377')

print(str(get_word_vector_glove('chomsky',glove_embeddings).shape) == '(300,)')
print(str(np.mean(get_word_vector_glove('chomsky',glove_embeddings))) == '-0.0065580728')
print(str(get_word_vector_glove('chomsky',glove_embeddings)[0]) == '-0.08470377')

True
True
True
True
True
True
True
True
True


In [6]:
def cosine_similarity(vec1:np.array, vec2:np.array):
    """
    Returns cosine similarity between two vectors
    """
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return dot_product / (norm_vec1 * norm_vec2)


In [13]:
print(str(cosine_similarity(np.array([1,2,3]), np.array([1,2, -3]))) == '-0.2857142857142857')
print(str(cosine_similarity(np.array([1,2,3]), np.array([1,-2, 3]))) == '0.42857142857142855')
print(str(cosine_similarity(np.array([1,2,3]), np.array([1,2, 3]))) == '1.0')

sims = {'hello hello': '1.0000001',
        'hello hey': '0.93034637',
        'hello hi': '0.9133941',
        'hello supercalifragilisticexpialidociou': '0.73929405',
        'hello chomsky': '0.73929405',
        'hello cat': '0.6564517'
       }

for key,val in sims.items():
    word1, word2 = key.split()
    res = str(cosine_similarity(get_word_vector_glove(word1,glove_embeddings),
                  get_word_vector_glove(word2,glove_embeddings)))
    print(res)
    print(abs(float(res) - float(val)) < 1e-4)
    #print(res == val)

True
True
True
0.9999999
True
0.93034625
True
0.9133939
True
0.7392941
True
0.7392941
True
0.65645164
True


In [24]:
def find_similar(word_vec:np.array, n:int, embeddings:dict, exclude:list):
    """
    Params:
        word_vec: a word embedding
        n: number of similar words to return
        embeddings: key word, value embedding
        exclude: words to be excluded from the output
        
    Returns:
        n words most similar to word; This does not include words in the exclude list
    """
    similarities = {}
    for word, vec in embeddings.items():
        if word not in exclude:
            sim = cosine_similarity(word_vec, vec)
            similarities[word] = sim
    sorted_similarities = sorted(similarities.items(), key=lambda item: item[1], reverse=True)
    return [ (word,float(sim)) for word, sim in sorted_similarities[:n]]
    

In [25]:

find_similar(get_word_vector_glove('hello',glove_embeddings), 10, glove_embeddings, exclude=['hello'])

## Expected output
# [('hey', 0.93034637),
#  ('hi', 0.9133941),
#  ('thank', 0.8759045),
#  ('!', 0.8671343),
#  ('thanks', 0.86197555),
#  ('dear', 0.85105646),
#  ('happy', 0.8466327),
#  ('welcome', 0.838622),
#  ('here', 0.82807016),
#  ('sorry', 0.8268465)]


[('hey', 0.9303462505340576),
 ('hi', 0.9133939146995544),
 ('thank', 0.8759043216705322),
 ('!', 0.8671342134475708),
 ('thanks', 0.8619754910469055),
 ('dear', 0.8510563373565674),
 ('happy', 0.8466325998306274),
 ('welcome', 0.8386217951774597),
 ('here', 0.8280700445175171),
 ('sorry', 0.8268464803695679)]

In [26]:
find_similar(get_word_vector_glove('hello',glove_embeddings), 10, glove_embeddings, exclude=[])

## Expected output
# [('hello', 1.0000001),
#  ('hey', 0.93034637),
#  ('hi', 0.9133941),
#  ('thank', 0.8759045),
#  ('!', 0.8671343),
#  ('thanks', 0.86197555),
#  ('dear', 0.85105646),
#  ('happy', 0.8466327),
#  ('welcome', 0.838622),
#  ('here', 0.82807016)]

[('hello', 0.9999998807907104),
 ('hey', 0.9303462505340576),
 ('hi', 0.9133939146995544),
 ('thank', 0.8759043216705322),
 ('!', 0.8671342134475708),
 ('thanks', 0.8619754910469055),
 ('dear', 0.8510563373565674),
 ('happy', 0.8466325998306274),
 ('welcome', 0.8386217951774597),
 ('here', 0.8280700445175171)]

### Part 1.3

Answer the following questions: 

1. What is the time complexity of `find_similar` if your vocab has `v` words, your embedding size is `m`, and you want to find `n` most similar words to the inputted word? 
- We loop through all the words in the vocab and for each word we compute the cosine similarity: $\mathcal{O}(vm)$
- Sorted: $\mathcal{O}(v\log v)$
- Overall: $\mathcal{O}(vm+v\log v) \sim \mathcal{O}(vm)$ if we have a huge $m$.

2. Consider a scenario (e.g., web application that displays similar words) where you might have to repeatedly run `find_similar`, say for `x` times. What are the benefits and challenges of pre-computing the similarity between all words? How might you overcome the challenges? 
- After we have a pre-computing similarities, we have a constant-time look up. However, we need $\mathcal{O}(v^2)$ space to store it. 
- Based on the actual scenario, we can choose a constant number $n$, not scaled with $v$. This will use only $\mathcal{O}(vn)$ space.

## Part 2: Computing similarity between `distilgpt` embeddings

### Part 2.1

In [27]:
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "distilgpt2"
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

embedding_matrix = model.get_input_embeddings().weight

/opt/anaconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/nlp/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [28]:
words = ['hello', 'cat', 'chomsky', 'supercalifragilisticexpialidocious']

for word in words:
    tokens = tokenizer.tokenize(word, add_special_tokens=False, return_tensors="pt")
    print(tokens)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    print(ids)
    embeds = embedding_matrix[ids]
    print(word, embeds.shape)

['hello']
[31373]
hello torch.Size([1, 768])
['cat']
[9246]
cat torch.Size([1, 768])
['ch', 'omsky']
[354, 37093]
chomsky torch.Size([2, 768])
['super', 'cal', 'if', 'rag', 'il', 'ist', 'ice', 'xp', 'ial', 'id', 'ocious']
[16668, 9948, 361, 22562, 346, 396, 501, 42372, 498, 312, 32346]
supercalifragilisticexpialidocious torch.Size([11, 768])


#### Answer the following question
How does `distilgpt` handle words that are not in its vocab? 


`distilgpt` uses a subword tokenizer, Byte-Pair Encoding (https://huggingface.co/distilbert/distilgpt2). For words not in its vocab, it will split it into subword tokens that are in the vocab. The each subword embedding compose the embedding matrix of the actual word.

### Part 2.2


In [33]:
def get_hf_wordvec(word, tokenizer, embedding_matrix):
    """
    Returns an embedding that is the average over all the sub-word tokens
    """
    tokens = tokenizer.tokenize(word)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    embeds = embedding_matrix[ids]
    return torch.mean(embeds, dim=0).detach().numpy()

In [34]:
print(str(get_hf_wordvec('hello', tokenizer, embedding_matrix).shape) =='(768,)')
print(str(np.mean(get_hf_wordvec('hello', tokenizer, embedding_matrix))) =='-0.0013018699')
print(str(get_hf_wordvec('hello', tokenizer, embedding_matrix)[0]) =='-0.029814368')

print(str(get_hf_wordvec('supercalifragilisticexpialidocious', tokenizer, embedding_matrix).shape)=='(768,)')
print(str(np.mean(get_hf_wordvec('supercalifragilisticexpialidocious', tokenizer, embedding_matrix)))=='-0.0007369095')
print(str(get_hf_wordvec('supercalifragilisticexpialidocious', tokenizer, embedding_matrix)[0]) =='-0.023058223')

print(str(get_hf_wordvec('chomsky', tokenizer, embedding_matrix).shape) =='(768,)')
print(str(np.mean(get_hf_wordvec('chomsky', tokenizer, embedding_matrix)))=='0.0015589814')
print(str(get_hf_wordvec('chomsky', tokenizer, embedding_matrix)[0]) =='0.026135625')

True
True
True
True
True
True
True
True
True


### Part 2.3

What are the limitations of using `embedding_matrix` in `find_similar`? 

Since `embedding_matrix` uses subword as the "keys", `find_similar` would return the similar "subwords" instead of the actual words we want.

### Part 2.4

In [35]:
def create_hf_embeddings(hf_model_name: str, vocab:set):
    """
    Returns dictionary. Key: words in the vocab; Value: word embeddings for the hf_model for the word
    """
    model = AutoModel.from_pretrained(hf_model_name)
    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    
    embedding_matrix = model.get_input_embeddings().weight
    
    embeddings = {}
    
    for word in vocab:
        word_vec = get_hf_wordvec(word, tokenizer, embedding_matrix)
        embeddings[word] = word_vec
        
    return embeddings
    

In [64]:
## Create embeddings using the vocab from glove_embeddings. Run sanity checks to make sure your function works. 
embeddings = create_hf_embeddings("distilgpt2", set(glove_embeddings.keys()))
print(len(embeddings.keys()) == len(glove_embeddings.keys()))

words = ['hello', 'cat', 'chomsky', 'supercalifragilisticexpialidocious']

for word in words:
    if word in embeddings:
        vec = embeddings[word]
        print(f'"{word}": Dimensions', vec.shape, "Mean:", np.mean(vec))
    else:
        print(f'"{word}": Missing embedding')

True
"hello": Dimensions (768,) Mean: -0.0013018699
"cat": Dimensions (768,) Mean: -0.0017775308
"chomsky": Missing embedding
"supercalifragilisticexpialidocious": Missing embedding


## Part 3: Using analogies to study encoding of gender

### Part 3.1

In [46]:
def compute_analogy(analogy:str, embeddings:dict, n:int):
    """
    Params:
        analogy: String of the format: a - b + c
        embeddings:  key word, value embedding
        n: number of cloest words to return
    Returns: 
        n cloest words to the resulting analogy embedding
    """
    a, b, c = analogy.split(' - ')[0], analogy.split(' - ')[1].split(' + ')[0], analogy.split(' + ')[1]
    vec_a = embeddings[a] if a in embeddings else np.mean(np.array(list(embeddings.values())), axis=0)
    vec_b = embeddings[b] if b in embeddings else np.mean(np.array(list(embeddings.values())), axis=0)
    vec_c = embeddings[c] if c in embeddings else np.mean(np.array(list(embeddings.values())), axis=0)
    
    analogy_vec = vec_a - vec_b + vec_c
    
    exclude = [a, b, c]
    
    return find_similar(analogy_vec, n, embeddings, exclude)
    


In [47]:
## Add Tests
#'king - man + woman' = 'queen'
print(compute_analogy("king - man + woman", glove_embeddings, 1))

[('queen', 0.884157657623291)]


### Part 3.2
Come up with at least 10 analogies that you think are important for testing how robustly gender is encoded in some embeddings. Justify why you picked the examples you did. 

1. "prince - man + woman" = "princess" (royal title)
2. "actor - man + woman" = "actress" (occupation)
3. "father - man + woman" = "mother" (family role)
4. "husband - man + woman" = "wife" (marriage role)
5. "brother - man + woman" = "sister" (family role - young)
6. "king - boy + girl" = "queen" (young gender nouns)
7. "king - male + female" = "queen" (neutral gender nouns)
8. "king - he + she" = "queen" (gender pronouns)
9. "queen - woman + man" = "king" (swap)
10. "he - man + woman" = "she" (pronoun test)

### Part 3.3
Using analogies from the previous part, test how robustly gender is encoded in `GLoVe` and `distilgpt` embeddings. Add as many code and markdown chunks as you would like. 

In [61]:
test_strs = ["prince - man + woman", "actor - man + woman", "father - man + woman", "husband - man + woman", "brother - man + woman",
             "king - boy + girl", "king - male + female", "king - he + she", "queen - woman + man", "he - man + woman"]
print("GloVe Embeddings")
for test_str in test_strs:
    print(test_str+" =")
    print(compute_analogy(test_str, glove_embeddings, 1))
print("\nDistilGPT2 Embeddings")
for test_str in test_strs:
    print(test_str+" =")
    print(compute_analogy(test_str, embeddings, 5))

GloVe Embeddings
prince - man + woman =
[('princess', 0.8398212790489197)]
actor - man + woman =
[('actress', 0.9177096486091614)]
father - man + woman =
[('mother', 0.9344622492790222)]
husband - man + woman =
[('wife', 0.9121500849723816)]
brother - man + woman =
[('sister', 0.9291958808898926)]
king - boy + girl =
[('queen', 0.8922580480575562)]
king - male + female =
[('queen', 0.8338760137557983)]
king - he + she =
[('queen', 0.9121057987213135)]
queen - woman + man =
[('king', 0.8834215402603149)]
he - man + woman =
[('she', 0.9539329409599304)]

DistilGPT2 Embeddings
prince - man + woman =
[('convince', 0.5500507354736328), ('province', 0.5024499297142029), ('printers', 0.4945780336856842), ('pristine', 0.4853276014328003), ('pricing', 0.4822002649307251)]
actor - man + woman =
[('act', 0.47990912199020386), ('acting', 0.46729233860969543), ('actress', 0.46015051007270813), ('female', 0.44012725353240967), ('interact', 0.43957221508026123)]
father - man + woman =
[('mother', 0.6

## Part 4 (optional): Using analogies to study encoding of other features

#### Feature: Tense of verbs
Tense captures temporal structure of the same event type but at different times. Encoding tense allows models to generalize across forms of the same lemma, improving downstream performance in translation, summarization, and language modeling.

In [80]:
test_strs = [
    "acted - past + present",
    "was - past + present",
    "was - past + now",
    "were - past + present",
    "ran - past + present",
    "swam - past + present",
    "read - past + present",
    "acted - ed + ing"
    ]
for test_str in test_strs:
    print(test_str+" =")
    print(compute_analogy(test_str, glove_embeddings, 3))
print("\nDistilGPT2 Embeddings")
for test_str in test_strs:
    print(test_str+" =")
    print(compute_analogy(test_str, embeddings, 3))

acted - past + present =
[('acting', 0.7927783131599426), ('manner', 0.7437383532524109), ('described', 0.7370191812515259)]
was - past + present =
[('described', 0.9033927321434021), ('by', 0.8858681321144104), ('latter', 0.8808571100234985)]
was - past + now =
[('had', 0.9233306646347046), ('came', 0.9181240200996399), ('did', 0.9166668057441711)]
were - past + present =
[('of', 0.8592160940170288), ('identified', 0.8577486872673035), ('described', 0.8494564294815063)]
ran - past + present =
[('called', 0.8302657008171082), ('by', 0.8283827900886536), ('described', 0.8193468451499939)]
swam - past + present =
[('moreover', 0.8578457236289978), ('furthermore', 0.8516912460327148), ('particular', 0.8349526524543762)]
read - past + present =
[('book', 0.8406516313552856), ('description', 0.8392257690429688), ('written', 0.8220885396003723)]
acted - ed + ing =
[('being', 0.7147788405418396), ('acting', 0.7067598104476929), ('that', 0.6794301867485046)]

DistilGPT2 Embeddings
acted - past

- Usefulness: Word analogies provide insight into what a model learns. They’re especially helpful for linear semantic relations.

- Limitations: However, analogies work only if the embedding space is approximately linear and meaning is static, which is true for GloVe but not for contextual models Contextual models (distilgpt) do not assign a single vector per word; their meaning depends on context. So analogy tests on token averages are often misleading. Features that depend on syntax, such as Tense we tested, are not easily probed by analogies.